# L03 · 가장 작은 RL: bandit

## Goal

- 탐색과 활용을 구분한다
- regret를 계산한다
- 표본 reward로 action estimate를 갱신한다

## Setup

이 cell은 CPU·seed·offline 상태와 split hash를 먼저 고정합니다. toy 연산은 결정론적인 CPU 연산만 쓰며, package trainer의 전역 결정론 기본값은 유지합니다.

In [1]:
import hashlib, json, os, platform, random, sys
from pathlib import Path
os.environ.setdefault("TORCH_DEVICE_BACKEND_AUTOLOAD", "0")
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the RL-study repository")
sys.path.insert(0, str(ROOT / "src"))
import torch
from rl_study import __version__
from rl_study.data import build_tiny_reasoning
from rl_study.runtime import resolve_device, seed_everything
# These notebooks use only deterministic CPU toy kernels.  PyTorch 2.13's global
# guard imports the full Inductor stack, so keep the package's strict default for
# trainers while avoiding that unrelated startup cost in fresh teaching kernels.
seed_everything(42, deterministic=False)
random.seed(42)
language = os.environ.get("RL_STUDY_NOTEBOOK_LANGUAGE", "ko")
resolution = resolve_device("cpu")
dataset = build_tiny_reasoning(seed=42)
config_hash = "sha256:" + hashlib.sha256(b"L03:toy:42").hexdigest()
print(f"lesson=L03 language={language} profile=toy")
print("seed=42 network_required=False deterministic_scope=seeded_cpu_toy")
print(f"python={platform.python_version()} rl_study={__version__} torch={torch.__version__}")
print(f"requested_device=cpu resolved_device={resolution.resolved} fallback_used={resolution.fallback_used}")
print(f"config_hash={config_hash} data_split_hash={dataset.split_hash}")

lesson=L03 language=ko profile=toy
seed=42 network_required=False deterministic_scope=seeded_cpu_toy
python=3.10.12 rl_study=0.1.0.dev0 torch=2.13.0
requested_device=cpu resolved_device=cpu fallback_used=False
config_hash=sha256:590ac6402f99d22939b7ae82eae358e15f3fe446668384655cf87d7b911bfaea data_split_hash=sha256:f238657bbf6c0a112debf7ef3ffafb452c14308dfb5ce57d9abe4f77ac1deedd


## Steps

### 1. 현재 위치와 핵심 식

⏱ 5분 · 1/3 section · [필수/CORE]

현재 위치: 확률·미분 → **bandit** → MDP

$$Q_{n+1}(a)=Q_n(a)+\frac{1}{N_n(a)}\left(R_n-Q_n(a)\right)$$

bandit에는 state transition이 없고 매 순간 arm 하나만 고릅니다. exploitation은 현재 최대 추정값을, exploration은 아직 모르는 arm을 시험합니다. regret는 실제 표본 손실이 아니라 최적 arm 기대보상과 선택 arm 기대보상의 누적 차이입니다.

### 2. 작은 숫자로 실행

⏱ 6분 · 2/3 section · [필수/CORE]

**먼저 예측:** 초기 추정값이 모두 0일 때 순수 greedy는 항상 좋은 arm을 찾을까요? 20초 동안 답을 적은 뒤 실행하세요.

<details><summary>정답 보기</summary>아닙니다. 초반 우연과 tie-breaking에 갇힐 수 있습니다. 작은 epsilon이 이 실패 경로를 끊습니다.</details>

In [2]:
from rl_study.envs import BernoulliBandit
def run_bandit(epsilon):
    env = BernoulliBandit(horizon=120)
    env.reset(seed=42)
    counts, estimates, regret = [0] * 5, [0.0] * 5, 0.0
    rng = random.Random(42)
    for step in range(120):
        explore = rng.random() < epsilon
        action = rng.randrange(5) if explore else max(range(5), key=estimates.__getitem__)
        result = env.step(action)
        counts[action] += 1
        estimates[action] += (result.reward - estimates[action]) / counts[action]
        regret += float(result.info["expected_regret"])
    return regret, counts, estimates
greedy_run = run_bandit(0.0)
exploring_run = run_bandit(0.1)
print({"greedy_regret": round(greedy_run[0], 2),
       "epsilon_regret": round(exploring_run[0], 2),
       "epsilon_counts": exploring_run[1]})

{'greedy_regret': 84.0, 'epsilon_regret': 28.25, 'epsilon_counts': [4, 3, 59, 1, 53]}


### 3. 구현 해부

⏱ 6분 · 3/3 section · [심화/DEEP DIVE]

**왜 이렇게 구현했나:** incremental mean은 전체 reward 기록 없이 같은 표본평균을 만듭니다. UCB나 Thompson sampling도 대안이지만, epsilon-greedy가 exploration 자체의 역할을 가장 투명하게 드러냅니다.

**흔한 함정:** sample reward로 regret를 계산하면 운 좋은 나쁜 arm이 음의 regret를 만들 수 있습니다. 환경이 제공한 expected regret를 별도로 누적합니다. 회귀 test: `test_bandit_seed_is_deterministic`.

**쉬어가기:** 지금 출력한 한 값만 설명할 수 있으면 다음 cell로 가세요.

## Checks

In [3]:
assert sum(exploring_run[1]) == 120 and exploring_run[0] >= 0.0
print("checks=passed")

checks=passed


**회상 문제:** exploration이 reward를 즉시 낮추면서도 장기 regret를 줄일 수 있는 이유는 무엇인가요? 1~2문장으로 답하세요.

## 내가 자주 틀리는 것

- loss가 유한하면 구현도 맞다고 생각한다.
- `terminated`와 `truncated`, prompt와 action을 합친다.
- 한 seed의 작은 결과를 알고리즘 순위로 확대한다.

## 60초 요약

- **실행 결론:** 고정 seed 출력에서 epsilon=0.1의 누적 regret가 순수 greedy보다 낮았습니다. 이는 이 한 환경의 실패 사례이며 보편적 우월성 주장이 아닙니다.
- 실제 확인: `test_bandit_seed_is_deterministic`.
- 출력은 고정 seed의 toy 실행이며 논문 규모 결과가 아닙니다.

## Next Steps

1. L04에서 state와 transition을 추가해 현재 action이 미래 reward까지 바꾸는 MDP로 갑니다.
2. `[필수/CORE]` assertion을 한 번 깨뜨리고 오류를 읽습니다.
3. package test를 열어 notebook의 작은 식과 production guard를 연결합니다.

## Sources

- `sutton-barto-rl2` — `docs/sources.yml`